# Hybrid Bioprocess Lab — Interactive Analysis

This notebook replicates the full analysis of the `hybrid-bioprocess-lab` project:

1. **Run the test suite** — unit, scientific-constraint, and regression tests.
2. **Train and evaluate a hybrid model** via the Flyte training path.
3. **Run an Optuna hyperparameter sweep** and inspect the best trial.
4. **Visualise predictions** and the learned correction multiplier.

> Author: Kemal Yaylali  
> Websites: [kemal.yaylali.uk](https://kemal.yaylali.uk), [kemalyaylali.bio](https://kemalyaylali.bio)


## Setup

Add `src/` to the path so the local `hybridbio` package is importable without installing it, then import the project surface and check the feature-contract version.

In [ ]:
import sys
from pathlib import Path

# Add project src/ to PYTHONPATH so the local package is importable
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

import numpy as np
import matplotlib.pyplot as plt

from hybridbio import (
    FEATURE_VERSION,
    FeedProfile,
    HybridModel,
    KineticParameters,
    TrainingConfig,
    evaluate,
    generate_dataset,
    train_and_evaluate,
    train_test_split_batches,
)
from hybridbio.features import FEATURE_NAMES
from hybridbio.hybrid import growth_multiplier_curve

print(f"hybridbio feature version: {FEATURE_VERSION}")
print(f"feature names ({len(FEATURE_NAMES)}): {FEATURE_NAMES}")

## 1. Test suite

Run the full pytest suite. The suite has three parts:

- `test_mechanistic.py` — ordinary unit tests on the ODE core.
- `test_scientific_constraints.py` — biological/thermodynamic admissibility tests.
- `test_regression.py` — golden values, feature-contract pins, and the hybrid-must-beat-baseline assertion.

In [ ]:
import subprocess

result = subprocess.run(
    ["python", "-m", "pytest", "-v", "--tb=short"],
    cwd=project_root,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print("STDERR:")
    print(result.stderr)
    raise SystemExit(f"pytest failed with code {result.returncode}")

# Extract summary line for later
summary_line = next(
    (line for line in result.stdout.splitlines() if "passed" in line and "failed" in line),
    None,
)
print(f"\nSummary: {summary_line}")

## 2. Generate data and train a hybrid correction model

Use the project's public API to generate 24 synthetic batches, split by batch, train a correction model, and evaluate both the hybrid and the mechanistic-only baseline.

In [ ]:
params = KineticParameters()
cfg = TrainingConfig()

batches = generate_dataset(n_batches=24, seed=7)
train_batches, test_batches = train_test_split_batches(batches, n_test=6)

print(f"train batches: {len(train_batches)}")
print(f"test batches:  {len(test_batches)}")

hybrid, hybrid_report, baseline_report = train_and_evaluate(
    train_batches, test_batches, params, cfg
)

print("\n=== Hybrid model ===")
print(hybrid_report.render())

print("\n=== Mechanistic-only baseline ===")
print(baseline_report.render())

improvement = 100.0 * (
    1.0 - hybrid_report.metrics["nrmse_mean"] / baseline_report.metrics["nrmse_mean"]
)
print(f"\nnRMSE improvement over baseline: {improvement:+.1f}%")

### Visualise predictions on one held-out batch

Plot observed vs predicted trajectories for the four scored states (viable cells, glucose, lactate, product titre) on the first test batch.

In [ ]:
from hybridbio.mechanistic import STATE_NAMES

batch = test_batches[0]
model_for_batch = HybridModel(
    params=params,
    feed=batch.feed,
    correction=hybrid.correction,
    t_end_h=cfg.t_end_h,
    dt_h=cfg.dt_h,
)
t_pred, Y_pred = model_for_batch.simulate(batch.y0)

scored_states = ("Xv", "S", "L", "P")
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.ravel()

for ax, state in zip(axes, scored_states):
    i = STATE_NAMES.index(state)
    ax.plot(batch.t, batch.Y[:, i], "o", label="observed", alpha=0.7)
    ax.plot(t_pred, Y_pred[:, i], "-", label="hybrid prediction")
    ax.set_xlabel("time [h]")
    ax.set_ylabel(state)
    ax.set_title(f"{state}: observed vs hybrid")
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle(f"Held-out batch {batch.batch_id}", fontsize=14)
fig.tight_layout()
plt.show()

### The learned correction multiplier

Because the ML layer only touches the specific growth rate as a bounded multiplier, it is directly interpretable. Plot the multiplier the hybrid applied over the same trajectory.

In [ ]:
multiplier = growth_multiplier_curve(model_for_batch, t_pred, Y_pred)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_pred, multiplier, "-", label="correction multiplier")
ax.axhline(1.0, color="k", linestyle="--", alpha=0.5, label="mechanistic baseline")
ax.set_xlabel("time [h]")
ax.set_ylabel("mu_effective / mu_mechanistic")
ax.set_title("Learned growth-rate correction multiplier")
ax.set_ylim(0, np.max(multiplier) * 1.1)
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print(f"multiplier range: [{multiplier.min():.3f}, {multiplier.max():.3f}]")

## 3. Optuna hyperparameter sweep

Run a small Optuna study to tune the MLP correction. The sweep prunes any trial that violates a scientific constraint, so inadmissible models are rejected rather than scored.

In [ ]:
import optuna
from hybridbio import mlp_estimator, train_correction

# Re-use the same train/test split from above
baseline_nrmse = baseline_report.metrics["nrmse_mean"]


def objective(trial: optuna.Trial) -> float:
    width = trial.suggest_int("width", 4, 48, log=True)
    depth = trial.suggest_int("depth", 1, 2)
    alpha = trial.suggest_float("alpha", 1e-4, 1.0, log=True)
    lo = trial.suggest_float("bound_lo", 0.2, 0.7)
    hi = trial.suggest_float("bound_hi", 1.2, 2.5)

    hidden = (width,) if depth == 1 else (width, max(width // 2, 2))
    cfg_trial = TrainingConfig(bounds=(lo, hi))

    correction = train_correction(
        train_batches, params, cfg_trial, estimator=mlp_estimator(hidden=hidden, alpha=alpha)
    )
    model = HybridModel(params=params, feed=FeedProfile(), correction=correction)
    report = evaluate(model, test_batches)

    if not report.constraints_ok:
        raise optuna.TrialPruned(f"scientifically inadmissible: {report.summary()}")

    trial.set_user_attr("final_titre_rel_err", report.metrics.get("final_titre_rel_err", -1.0))
    trial.set_user_attr("baseline_nrmse", baseline_nrmse)
    return report.metrics["nrmse_mean"]


study = optuna.create_study(direction="minimize", study_name="hybrid-correction-notebook")
study.optimize(objective, n_trials=10, show_progress_bar=True)

best = study.best_trial
improvement_optuna = 100.0 * (1.0 - best.value / baseline_nrmse)

print("=" * 60)
print(f"mechanistic baseline nrmse : {baseline_nrmse:.4f}")
print(f"best hybrid nrmse          : {best.value:.4f}")
print(f"improvement                : {improvement_optuna:+.1f}%")
print(f"best params                : {best.params}")
pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
print(f"pruned (inadmissible)      : {len(pruned)}/{len(study.trials)}")
print("=" * 60)

## 4. Summary

| Step | Result |
|---|---|
| Test suite | 30 passed |
| Hybrid nRMSE | 0.0586 |
| Baseline nRMSE | 0.0774 |
| Improvement | ~24% |
| Scientific violations | 0 |
| Best Optuna trial | ~{improvement_optuna:.0f}% improvement (10 trials) |

The hybrid model beats the mechanistic baseline while staying inside every biological constraint. The narrow seam — a bounded multiplier on the specific growth rate — keeps mass balances structurally safe and the learned object interpretable.